# Multi-agent style analysis (outputs/new_eval)

Same analysis as `analyze_multi_styles.ipynb`, but reads the **`outputs/new_eval`** layout, where
each result folder is named `<model-key>_<style>` and holds exactly **one** run directory
(`multi__…` for styles, `single__…` for the baseline):

```
outputs/new_eval/
  dsv4_single/    single__DeepSeek-V4-Pro__single__…/   traj/*.json  eval.json
  dsv4_neutral/   multi__DeepSeek-V4-Pro__…/            traj/*.json  eval.json
  dsv4_delegate/  multi__…/
  dsv4_solo/      multi__…/
  glm52_single/   …
  …
```

**Table 1** - single acc per model.
**Table 2** - multi metrics per (model, style): avg sub-agents / zero-deleg% / multi acc / diff vs single.
**Table 3** - platform-level delegation (integer counts, run-matched).
**Table 4** - effort vs quality on the orchestrator's OWN (self-handled) platforms.
**Table 5** - total in/out context tokens, coordination share, and a no-openapi estimate.

Model/style folders are **auto-discovered**. Run from the repo root (paths are relative).

In [ ]:
# ==== paths ====
# each folder under EVAL_ROOT is '<model-key>_<style>' and holds ONE run dir.
#   style == 'single' -> single__ baseline ;  otherwise -> multi__ run for that style.
import os as _os
# resolve 'outputs/new_eval' whether the notebook runs from repo root or from analysis/
EVAL_ROOT = next((p for p in ('outputs/new_eval', '../outputs/new_eval')
                  if _os.path.isdir(p)), 'outputs/new_eval')
# styles to treat as the single baseline (not a multi style)
SINGLE_STYLE = 'single'

In [2]:
import json, glob, os, re
from collections import defaultdict
try:
    import pandas as pd
    HAVE_PD = True
except ImportError:
    HAVE_PD = False

def _parts(f):
    # filename = <task(5 chunks)>-<run_idx>.json  ->  (task_id, run_idx)
    b = os.path.basename(f)[:-5].split('-')
    return '-'.join(b[:5]), (b[5] if len(b) > 5 else '0')

def _tid(f):
    return _parts(f)[0]

# http base_url shows up in BOTH call formats the harness accepts & executes:
#   XML  : <parameter=base_url>http://...     (Qwen always; GLM/Kimi sometimes)
#   JSON : {... "base_url": "http://..."}      (GLM ~30% / Kimi ~50% of calls)
# This one regex catches both (the separator class eats '>' or '": ').
_BASE_URL_RE = re.compile(r'base_url["\s>:]*(http://[^"<\s]+)')

def _base_urls(content):
    return [u.strip().rstrip('/') for u in _BASE_URL_RE.findall(content or '')]

def run_metrics(run_dir):
    # returns n_tasks, n_runs, avg_sub, zero_deleg%, acc (mean over tasks)
    fs = glob.glob(os.path.join(run_dir, 'traj', '*.json'))
    if not fs:
        return None
    by_task = defaultdict(list); subs = []
    for f in fs:
        t = json.load(open(f))
        by_task[_tid(f)].append(t.get('acc', 0))
        roles = [e.get('role', '') for e in t.get('events', [])]
        subs.append(len(set(r for r in roles if r.startswith('subagent_') and not r.endswith(':user'))))
    ta = {k: sum(v) / len(v) for k, v in by_task.items()}
    return dict(n_tasks=len(ta), n_runs=len(fs),
                avg_sub=sum(subs) / len(subs),
                zero=100 * sum(1 for s in subs if s == 0) / len(subs),
                acc=sum(ta.values()) / len(ta))

def model_of(d):
    # multi__<model>__<sub>__<ts>  or  single__<model>__single__<ts>
    return os.path.basename(d).split('__')[1]

def _inner_run_dir(folder):
    # the single run directory inside a '<model>_<style>' folder
    inner = sorted(glob.glob(os.path.join(folder, 'single__*')) +
                   glob.glob(os.path.join(folder, 'multi__*')))
    return inner[0] if inner else None

def discover():
    # -> (single_dirs {model: run_dir}, multi_dirs {(model, style): run_dir})
    single_dirs, multi_dirs = {}, {}
    for folder in sorted(glob.glob(os.path.join(EVAL_ROOT, '*'))):
        if not os.path.isdir(folder):
            continue
        style = os.path.basename(folder).rsplit('_', 1)[-1]
        rd = _inner_run_dir(folder)
        if not rd:
            continue
        mo = model_of(rd)
        if style == SINGLE_STYLE:
            single_dirs[mo] = rd
        else:
            multi_dirs[(mo, style)] = rd
    return single_dirs, multi_dirs

def show(rows):
    # pandas DataFrame if available, else aligned text
    if HAVE_PD:
        from IPython.display import display
        display(pd.DataFrame(rows)); return
    if not rows:
        print('(no data)'); return
    cols = list(rows[0].keys())
    w = {c: max(len(str(c)), *(len(str(r[c])) for r in rows)) for c in cols}
    print('  '.join(str(c).ljust(w[c]) for c in cols))
    for r in rows:
        print('  '.join(str(r[c]).ljust(w[c]) for c in cols))

# ---- platform-level helpers (Table 3) ----
def single_by_run(single_dir):
    # (task, run_idx) -> {platform: status}  (lets us match multi run i to single run i)
    d = {}
    for f in glob.glob(os.path.join(single_dir, 'traj', '*.json')):
        tid, ri = _parts(f)
        d[(tid, ri)] = json.load(open(f)).get('verifier_results') or {}
    return d

def platform_counts(multi_dir, single_runs):
    # integer counts over every scored platform-instance (per run; no dedup, no average).
    # single is matched by the SAME (task, platform, run_idx) slot.
    tot = deleg = nself = 0
    dp = dps = sp = sps = 0   # multi/single passes for delegated / self platforms
    for f in glob.glob(os.path.join(multi_dir, 'traj', '*.json')):
        tid, ri = _parts(f); t = json.load(open(f))
        ev = t.get('events', []); vr = t.get('verifier_results') or {}
        sysmsg = [e['content'] for e in ev if e.get('role') == 'orchestrator']
        if not sysmsg:
            continue
        # platform name -> url, parsed from the orchestrator's platform list
        url2name = {u: n.strip() for n, u in re.findall(r'- ([^(\n]+?) \((http://[^)]+)\)', sysmsg[0])}
        # a platform is 'delegated' if some sub-agent called its url (either call format)
        deleg_set = set()
        for e in ev:
            r = e.get('role', '')
            if r.startswith('subagent_') and not r.endswith(':user'):
                for u in _base_urls(e.get('content', '')):
                    nm = url2name.get(u)
                    if not nm:
                        for uu, nn in url2name.items():
                            if u.startswith(uu.rstrip('/')):
                                nm = nn; break
                    if nm:
                        deleg_set.add(nm)
        svr = single_runs.get((tid, ri), {})
        for p, s in vr.items():
            if s == 'skip':
                continue
            tot += 1
            mp = (s == 'complete'); sp_single = (svr.get(p) == 'complete')
            if p in deleg_set:
                deleg += 1; dp += mp; dps += sp_single
            else:
                nself += 1; sp += mp; sps += sp_single
    return dict(total=tot, delegated=deleg, deleg_pass=dp, deleg_pass_single=dps,
                self=nself, self_pass=sp, self_pass_single=sps)

SINGLE_DIRS, MULTI_DIRS = discover()
STYLES = sorted({st for (_m, st) in MULTI_DIRS})
print('single models :', sorted(SINGLE_DIRS))
print('multi styles  :', STYLES)
print('multi (model,style) count:', len(MULTI_DIRS))

single models : []
multi styles  : []
multi (model,style) count: 0


## Table 1 - single

In [3]:
single_acc = {}
single_rows = []
for mo, d in sorted(SINGLE_DIRS.items()):
    m = run_metrics(d)
    if not m:
        continue
    single_acc[mo] = m['acc']
    single_rows.append({'model': mo, 'tasks': f"{m['n_tasks']}t/{m['n_runs']}",
                        'single_acc': round(m['acc'], 3)})
show(single_rows)

""


## Table 2 - multi by (model, style)

In [4]:
rows = []
for (mo, style), d in MULTI_DIRS.items():
    m = run_metrics(d)
    if not m:
        continue
    sa = single_acc.get(mo, float('nan'))
    rows.append({'model': mo, 'style': style,
                 'tasks': f"{m['n_tasks']}t/{m['n_runs']}",
                 'avg_sub': round(m['avg_sub'], 2),
                 'zero_deleg%': round(m['zero']),
                 'multi_acc': round(m['acc'], 3),
                 'single_acc': round(sa, 3),
                 'diff': round(m['acc'] - sa, 3)})
rows.sort(key=lambda r: (r['model'], r['style']))
show(rows)

""


## Table 3 - platform-level delegation (integer counts)

Per (model, style), over all scored platform-instances (skip excluded; counted per run, no dedup):
- `total_plat` = total platform-instances
- `delegated` / `self` = handed to a sub-agent / handled by the orchestrator
- `deleg_pass` / `self_pass` = how many pass verifier in **multi**
- `deleg_pass_single` / `self_pass_single` = how many the SAME (task, platform, run) slots pass in **single**
- `diff` = total passed in multi - total passed in single  (>0 = multi passes more platforms)

In [5]:
# single runs indexed by (task, run_idx), per model
single_runs_by_model = {mo: single_by_run(d) for mo, d in SINGLE_DIRS.items()}

rows3 = []
for (mo, style), d in MULTI_DIRS.items():
    if not glob.glob(os.path.join(d, 'traj', '*.json')):
        continue
    c = platform_counts(d, single_runs_by_model.get(mo, {}))
    diff = (c['deleg_pass'] + c['self_pass']) - (c['deleg_pass_single'] + c['self_pass_single'])
    rows3.append({'model': mo, 'style': style,
                  'total_plat': c['total'],
                  'delegated': c['delegated'],
                  'deleg_pass': c['deleg_pass'],
                  'deleg_pass_single': c['deleg_pass_single'],
                  'self': c['self'],
                  'self_pass': c['self_pass'],
                  'self_pass_single': c['self_pass_single'],
                  'diff': diff})
rows3.sort(key=lambda r: (r['model'], r['style']))
show(rows3)

""


## Table 4 - effort vs quality on the orchestrator's OWN platforms

For the platforms the orchestrator handled itself (not delegated), compare its **effort** and
**pass rate** against single on the SAME (task, platform, run) slots:
- `self_n` = self-handled platform-instances
- `m_calls` / `s_calls` = avg http calls per platform in **multi (orchestrator)** / **single**  (effort)
- `m_pass%` / `s_pass%` = pass rate on those platforms in multi / single  (quality)
- `d_pp` = `m_pass% - s_pass%`  (negative = orchestrator does worse on its own platforms than single)

Reads the "divided-attention" tax: if `m_calls >= s_calls` but `d_pp < 0`, the orchestrator spent
**equal-or-more** effort yet did **worse** — degradation is from distraction, not from under-attempting.

In [6]:
# http calls to each platform, by the given roles (uses _base_urls -> both call formats)
def _plat_calls(events, url2name, roles):
    cnt = {}
    for e in events:
        if e.get('role') not in roles:
            continue
        for u in _base_urls(e.get('content', '')):
            nm = url2name.get(u)
            if not nm:
                for uu, nn in url2name.items():
                    if u.startswith(uu.rstrip('/')):
                        nm = nn; break
            if nm:
                cnt[nm] = cnt.get(nm, 0) + 1
    return cnt

def single_effort(single_dir):
    # (task, run, platform) -> (n_http_calls, passed)
    S = {}
    for f in glob.glob(os.path.join(single_dir, 'traj', '*.json')):
        tid, ri = _parts(f); t = json.load(open(f))
        ev = t.get('events', []); vr = t.get('verifier_results') or {}
        sm = [e['content'] for e in ev if e.get('role') == 'agent']
        u2n = {u: n.strip() for n, u in re.findall(r'- ([^(\n]+?) \((http://[^)]+)\)', sm[0])} if sm else {}
        cnt = _plat_calls(ev, u2n, {'agent'})
        for p, s in vr.items():
            if s == 'skip':
                continue
            S[(tid, ri, p)] = (cnt.get(p, 0), s == 'complete')
    return S

single_effort_by_model = {mo: single_effort(d) for mo, d in SINGLE_DIRS.items()}

rows4 = []
for (mo, style), d in MULTI_DIRS.items():
    if not glob.glob(os.path.join(d, 'traj', '*.json')):
        continue
    S = single_effort_by_model.get(mo, {})
    m_calls = s_calls = m_pass = s_pass = n = 0
    for f in glob.glob(os.path.join(d, 'traj', '*.json')):
        tid, ri = _parts(f); t = json.load(open(f))
        ev = t.get('events', []); vr = t.get('verifier_results') or {}
        sm = [e['content'] for e in ev if e.get('role') == 'orchestrator']
        if not sm:
            continue
        url2name = {u: nm.strip() for nm, u in re.findall(r'- ([^(\n]+?) \((http://[^)]+)\)', sm[0])}
        # delegated platforms (excluded from self)
        deleg = set()
        for e in ev:
            r = e.get('role', '')
            if r.startswith('subagent_') and not r.endswith(':user'):
                for u in _base_urls(e.get('content', '')):
                    nm = url2name.get(u)
                    if not nm:
                        for uu, nn in url2name.items():
                            if u.startswith(uu.rstrip('/')):
                                nm = nn; break
                    if nm:
                        deleg.add(nm)
        ocnt = _plat_calls(ev, url2name, {'orchestrator'})
        for p, s in vr.items():
            if s == 'skip' or p in deleg:
                continue
            if (tid, ri, p) not in S:
                continue
            sc, scomp = S[(tid, ri, p)]
            n += 1
            m_calls += ocnt.get(p, 0); s_calls += sc
            m_pass += (s == 'complete'); s_pass += scomp
    if n == 0:
        continue
    rows4.append({'model': mo, 'style': style, 'self_n': n,
                  'm_calls': round(m_calls / n, 1), 's_calls': round(s_calls / n, 1),
                  'm_pass%': round(100 * m_pass / n), 's_pass%': round(100 * s_pass / n),
                  'd_pp': round(100 * m_pass / n - 100 * s_pass / n)})
rows4.sort(key=lambda r: (r['model'], r['style']))
show(rows4)

""


## Table 5 - total in/out context (tokens), coordination share, and a no-openapi estimate

Per (model, style), with a `single` baseline row:
- `orch_in` / `orch_out` = orchestrator (single: the lone agent) input / output tokens, per run
- `sub_in` / `sub_out` = total tokens consumed by all sub-agents (0 for single)
- `total_in` / `total_out` = orch + sub, per run
- `in_no_oa` / `out_no_oa` = estimated total tokens if sub-agents/agent did NOT fetch `GET /openapi.json`.
  openapi is a tool **response** (input side) re-fed every turn, so `out_no_oa` ~= `total_out`; `in_no_oa`
  is `total_in` x (cumulative input chars with openapi responses zeroed / with them kept).
- `out_coord%` = share of the orchestrator's OUTPUT that is coordination (spawn briefs + get_task_results calls)
- `inj_coord%` = share of injected tool-results that is coordination (sub-agent summaries) vs platform http responses

Tokens come from the `tokens` field (per-turn aggregates, not splittable per tool), so `coord%` and the
`no_oa` estimate are computed at the **character** level. `in_no_oa` is a counterfactual (zeroing the openapi
re-feeding), not a re-run. Watch `sub_in` and `in_no_oa`: most of the input cost — in single AND multi —
is the openapi schema being re-fed every turn.

In [7]:
# classify a tool call by its argument keys (no name stored, so infer from keys)
def _tool_kind(a):
    if not isinstance(a, dict):
        return 'other'
    if 'base_url' in a:
        return 'http'                       # platform work
    if 'description' in a or 'return_requirements' in a:
        return 'spawn'                      # coordination
    if 'task_ids' in a:
        return 'get_results'                # coordination
    return 'other'

def _is_openapi(s):
    s = str(s)
    return '/openapi.json' in s[:60] or '"openapi"' in s[:200]

def _ctx_cum(events_in_order, brief_chars):
    # cumulative INPUT chars (re-feeding) with vs without openapi responses, for ONE agent context.
    # events_in_order[0] is the system prompt (no tool calls); the rest are turns.
    # context at turn t = system + brief + everything generated/returned before t (re-fed each turn).
    if not events_in_order:
        return 0.0, 0.0
    preamble = len(events_in_order[0].get('content', '') or '') + brief_chars
    accw = acco = preamble
    cum_w = cum_o = 0.0
    for e in events_in_order[1:]:
        cum_w += accw; cum_o += acco
        c = len(e.get('content', '') or '')
        tcs = e.get('tool_calls') or []; trs = e.get('tool_responses') or []
        tcl = sum(len(json.dumps(x, ensure_ascii=False)) for x in tcs)
        trw = sum(len(str(x)) for x in trs)
        tro = sum(0 if _is_openapi(x) else len(str(x)) for x in trs)
        accw += c + tcl + trw; acco += c + tcl + tro
    return cum_w, cum_o

def context_tokens(run_dir, role):
    # per-run averages: orch/sub in/out tokens (from `tokens`), totals, a no-openapi estimate,
    # and char-based coordination share of OUTPUT / INJECTED tool-results.
    oin = oout = sin = sout = 0.0
    out_tot = out_coord = inj_plat = inj_coord = 0.0
    cum_w = cum_o = 0.0          # cumulative input chars with / without openapi (all contexts)
    n = 0
    for f in glob.glob(os.path.join(run_dir, 'traj', '*.json')):
        t = json.load(open(f)); tk = t.get('tokens') or {}
        oin += (tk.get('orch') or {}).get('in', 0); oout += (tk.get('orch') or {}).get('out', 0)
        sin += (tk.get('sub') or {}).get('in', 0); sout += (tk.get('sub') or {}).get('out', 0)
        # group events into per-actor contexts (+ their briefs from the :user events)
        ctx = defaultdict(list); briefs = defaultdict(float)
        for e in t.get('events', []):
            r = e.get('role', '')
            if r.endswith(':user'):
                briefs[r[:-5]] += len(e.get('content', '') or '')
            else:
                ctx[r].append(e)
        for actor, evs in ctx.items():
            if actor == role or actor.startswith('subagent_'):
                w, o = _ctx_cum(evs, briefs.get(actor, 0.0)); cum_w += w; cum_o += o
        # coordination share on the main role's output / injected results
        for e in ctx.get(role, []):
            out_tot += len(e.get('content', '') or '')
            tcs = e.get('tool_calls') or []; trs = e.get('tool_responses') or []
            for i, tc in enumerate(tcs):
                k = _tool_kind(tc); cl = len(json.dumps(tc, ensure_ascii=False))
                rl = len(str(trs[i])) if i < len(trs) else 0
                out_tot += cl
                if k in ('spawn', 'get_results'):
                    out_coord += cl
                if k == 'http':
                    inj_plat += rl
                elif k in ('spawn', 'get_results'):
                    inj_coord += rl
        n += 1
    if n == 0:
        return None
    inj = inj_plat + inj_coord
    ratio = cum_o / cum_w if cum_w else 1.0           # fraction of input left after removing openapi
    total_in = (oin + sin) / n; total_out = (oout + sout) / n
    return dict(orch_in=round(oin / n), orch_out=round(oout / n),
                sub_in=round(sin / n), sub_out=round(sout / n),
                total_in=round(total_in), total_out=round(total_out),
                in_no_oa=round(total_in * ratio),     # est. input tokens without openapi discovery
                out_no_oa=round(total_out),           # openapi is input-side only -> output ~unchanged
                out_coord_pct=round(100 * out_coord / out_tot) if out_tot else 0,
                inj_coord_pct=round(100 * inj_coord / inj) if inj else 0)

rows5 = []
for mo, d in sorted(SINGLE_DIRS.items()):
    c = context_tokens(d, 'agent')
    if c:
        rows5.append({'model': mo, 'style': 'single', **c})
for (mo, style), d in MULTI_DIRS.items():
    c = context_tokens(d, 'orchestrator')
    if c:
        rows5.append({'model': mo, 'style': style, **c})
_order = {'single': 0, 'neutral': 1, 'delegate': 2, 'solo': 3}
rows5.sort(key=lambda r: (r['model'], _order.get(r['style'], 9)))
show(rows5)

""
